In [1]:
!pip install transformers faiss-cpu datasets sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.7 MB/s eta 0:00:00


#These libraries include:

Transformers: For handling both the retriever and generator models.
Faiss: For efficient similarity search and clustering of dense vectors.
Datasets: For managing datasets.
Sentence-Transformers: For creating embeddings.

In [2]:
from datasets import Dataset
corpus = [
    "The Eiffel Tower is located in Paris.",
    "The Great Wall of China is visible from space.",
    "Python is a widely used programming language."
]
dataset = Dataset.from_dict({"text": corpus})

In [3]:
dataset

Dataset({
    features: ['text'],
    num_rows: 3
})

In [4]:
from transformers import AutoTokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
def preprocess(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)
tokenized_corpus = dataset.map(preprocess, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [6]:
from sentence_transformers import SentenceTransformer
import faiss

In [7]:
import numpy as np

In [8]:
retriever_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
#corpus_embeddings = retriever_model.encode(corpus, convert_to_tensor=True)
corpus_embeddings = retriever_model.encode(corpus, convert_to_tensor=True)

# Convert to numpy
corpus_embeddings = corpus_embeddings.cpu().numpy()

# Normalize
faiss.normalize_L2(corpus_embeddings)

# Create index
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

# index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
# faiss.normalize_L2(corpus_embeddings)
# index.add(corpus_embeddings.cpu().numpy())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
def retrieve(query, top_k=3):
    query_embedding = retriever_model.encode([query], convert_to_tensor=True)
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding.cpu().numpy(), top_k)
    return [corpus[i] for i in I[0]]

In [10]:
def retrieve(query, k=3):
    # Encode query as numpy
    query_embedding = retriever_model.encode(query, convert_to_numpy=True)

    # FAISS expects 2D array
    query_embedding = query_embedding.reshape(1, -1)

    # Normalize if using cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search
    distances, indices = index.search(query_embedding, k)

    return [corpus[i] for i in indices[0]]

In [11]:
query = "Where is the Eiffel Tower?"
retrieved_docs = retrieve(query)
print("Retrieved Documents:", retrieved_docs)

Retrieved Documents: ['The Eiffel Tower is located in Paris.', 'The Great Wall of China is visible from space.', 'Python is a widely used programming language.']


In [12]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [13]:
generator_model = T5ForConditionalGeneration.from_pretrained('t5-small')
generator_tokenizer = T5Tokenizer.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [14]:
def generate_answer(query, retrieved_docs):
    context = " ".join(retrieved_docs)
    input_text = f"question: {query} context: {context}"
    inputs = generator_tokenizer(input_text, return_tensors='pt')
    outputs = generator_model.generate(inputs['input_ids'], max_length=50)
    answer = generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer
answer = generate_answer(query, retrieved_docs)
print("Generated Answer:", answer)

Generated Answer: Paris


In [15]:
query = "What is Python?"
retrieved_docs = retrieve(query)
answer = generate_answer(query, retrieved_docs)
print(f"Query: {query}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Generated Answer: {answer}")

Query: What is Python?
Retrieved Documents: ['Python is a widely used programming language.', 'The Great Wall of China is visible from space.', 'The Eiffel Tower is located in Paris.']
Generated Answer: a widely used programming language


In [16]:
filtered_docs = [doc for doc in retrieved_docs if len(doc.split()) > 5]
answer = generate_answer(query, filtered_docs)
print(f"Filtered Answer: {answer}")

Filtered Answer: a widely used programming language
